# **Ch 6. Building a Stock Price Prediction System**

## <span style="color: blue;">**1. Prompt**</span>

<span style="color: blue;">**I want to build a system that estimates the probability of a stock price increase in real time using the following data.**</span>

![Stock](https://raw.githubusercontent.com/pilsunchoi/images4/main/6-1.png)

Source: Two screenshots from the Mirae Asset Securities mobile app

## **Claude: Idea**

I identified the order book and 1-minute candlestick chart data from the images. I will build a dashboard that estimates the probability of a price increase in real time by combining order flow imbalance, price momentum, trading volume, and other factors. Based on the uploaded order book and chart data, I have created a real-time probability estimation system for price increases.

**Estimation Model Structure** — Weighted sum of 6 factors:

| Factor | Weight | Description |
|---|---|---|
| Order book imbalance (overall) | 15% | Total bid volume vs. ask volume ratio |
| Order book imbalance (near price) | 25% | Imbalance of the 3 price levels nearest to the current price (most immediate signal) |
| Price momentum | 20% | Rate of price change over the last 5 candles |
| Volume signal | 15% | Proportion of volume on bullish candles + volume expansion |
| RSI signal | 10% | Overbought/oversold contrarian signal |
| Spread pressure | 15% | Best bid-ask spread + volume ratio |

In the dashboard below, pressing the **"Start Simulation"** button generates random ticks, updating the order book, chart, and probabilities every 1.2 seconds. To use with a real API, simply replace the simulation component with live data feeds.

> 🔗 **Stock Price Prediction System: Binary Model Dashboard** (https://claude.ai/public/artifacts/b52ce4f8-90bb-4782-9202-e778a5661dca)

![Stock](https://raw.githubusercontent.com/pilsunchoi/images4/main/6-2.png)

## <span style="color: blue;">**2. Rationale for Weight Settings**</span>

## **Claude: Response**

Good question. Let me explain the rationale behind each factor's weight.

**Near-Price Order Book Imbalance (25%) — Highest Weight**

The bid-ask volume difference at the 3 price levels immediately above and below the current price has the most direct impact on ultra-short-term price direction. While orders at distant price levels are frequently canceled or modified, near-price orders have a higher probability of being executed. In your uploaded data, the difference between the ask volume just above the current price (161,100: 15,800) and the bid volume just below (160,900: 11,965) suggests the short-term direction.

**Price Momentum (20%)**

Looking at the 1-minute chart, the pattern shows a sharp rally from 158,800 at 18:13 to 161,600 at 19:08, followed by consolidation. Whether this momentum continues or fades is a key variable determining the next move, which is why it was assigned the second-highest weight.

**Overall Order Book Imbalance (15%) & Spread Pressure (15%)**

The overall order book imbalance (ask 254K vs. bid 159K) shows the big-picture supply and demand, but distant price levels contain many spoofed orders, making them less reliable than near-price levels. Spread pressure also serves as a supplementary confirmation signal, so both were set at 15%.

**Volume Signal (15%)**

Following the principle that "price is confirmed by volume," this factor examines whether volume is concentrated on bullish candles. However, at the 1-minute candle level, noise is significant, so it was weighted lower than momentum.

**RSI Signal (10%) — Lowest Weight**

RSI is a lagging indicator, and since it is calculated based on 14 candles, it reacts slowly on 1-minute charts. It provides meaningful contrarian signals only in extreme overbought/oversold zones, so it was assigned a supplementary role at 10%.

---

To be candid, these weights are initial values reflecting conventional wisdom from market microstructure research. For more precise use in practice, it would be far more accurate to run backtests on historical data to find optimal weights, or to train a model such as logistic regression or XGBoost. If desired, I can also modify the UI to allow direct weight adjustment via sliders.

## <span style="color: blue;">**3. Is (1 - P(Up)) the Probability of Decline?**</span>

## **Claude: Response**

Strictly speaking, no.

The next price movement has three possible outcomes — up, unchanged, or down — so `1 - P(up)` = `P(unchanged) + P(down)`. The current model does not distinguish between these and treats the output as a binary score. For example, if the estimated probability of an increase is 40%, it is impossible to tell whether the remaining 60% represents a likely decline or a likely flat market.

There are three directions for improvement.

First, switching to a 3-class model. This would estimate separate probabilities for up, unchanged, and down, outputting them in a softmax form so they sum to 100%. Example: Up 35%, Unchanged 40%, Down 25%.

Second, introducing a threshold. Define movements within ±0.1% of the current price as unchanged, and classify only movements beyond that range as up or down.

Third, switching to expected return. Instead of probabilities, estimating the "expected price change over the next N minutes" can express both direction and magnitude simultaneously.

Shall I upgrade to a 3-class model?

## <span style="color: blue;">**4. Model Upgrade: 3-Class Classification**</span>

## **Claude: Work**

I will rewrite the entire file as a 3-class model. The key changes are softmax-based 3-class probabilities, introduction of an unchanged-zone threshold, and a 3-color probability trend chart. The upgrade to a 3-class model is complete. The major changes are as follows.

**Model Structure Change**

Instead of the previous binary score (0–100%), the model first computes a directional score from the weighted sum of 6 factors, then combines it with a volatility indicator to produce 3 logits. The up-logit increases when the directional score is positive, the down-logit increases when it is negative, and the unchanged-logit strengthens when volatility is low and directionality is weak. Softmax is applied to these 3 logits to convert them into a probability distribution summing to 100%.

**Unchanged Classification Logic**

Unchanged strength = `(1 - volatility) × 0.7 + (1 - directional strength) × 0.3`. This formula naturally raises the unchanged probability in periods where recent candle ranges are small and factors conflict, resulting in weak directionality.

**UI Changes**

The gauge chart has been replaced with a 3-class bar chart, and the probability trend chart now shows 3 separate lines (green/yellow/red). Directional score, volatility level, and RSI are displayed as separate metric cards.

> 🔗 **Stock Price Prediction System: 3-Class Model Dashboard** (https://claude.ai/public/artifacts/159d4993-6fdd-4468-8802-d5d25f6026f8)

![Stock](https://raw.githubusercontent.com/pilsunchoi/images4/main/6-3.png)